# IMPORT

In [106]:
import os
import json

import pandas as pd
import numpy as np

from collections import Counter

# LOAD DATA

In [107]:
sentiment_news_path = r"F:\UNIVERSITY\Project\Sentiment-Analysis-Airflow\Financial-Sentiment-Analysis\projects\data\sentiment_finance_news_with_scores.csv"
history_path = r"F:\UNIVERSITY\Project\Sentiment-Analysis-Airflow\Financial-Sentiment-Analysis\projects\data\history.csv"

In [108]:
sentiment_news_df = pd.read_csv(sentiment_news_path)
history_df = pd.read_csv(history_path)

In [109]:
history_df["time"] = pd.to_datetime(history_df["time"])
sentiment_news_df["posted_date"] = pd.to_datetime(sentiment_news_df["posted_date"])

In [110]:
history_df["time"].min()

Timestamp('2021-12-01 00:00:00')

# FUNCTION

In [111]:
def set_sentiment(labels, scores):
    def get_top(labels, scores):
        top = 0
        dic_count = {}
        dic_pair = {}
        for label, score in zip(labels, scores):
            if label not in dic_count:
                dic_count[label] = 0
                dic_pair[label] = []
            dic_count[label] += 1
            dic_pair[label].append((label, score))
            if dic_count[label] > top:
                top = dic_count[label]
        
        top_ids = [k for k, v in dic_count.items() if v==top]
        top_pairs = [dic_pair[id] for id in top_ids]
        return top_ids, top_pairs

    labels = [int(label) for label in labels.split(" ")]
    scores = [float(score) for score in scores.split(" ")]
    top_ids, top_pairs = get_top(labels, scores)

    # Filter
    if len(top_pairs)==0:
        return pd.Series([None, None])
    if len(top_pairs)==1:
        return pd.Series([top_ids[0], sum([score for label, score in top_pairs[0]])])
    
    top_scores = [sum([score for label, score in pair]) for pair in top_pairs]
    highest_index = np.argmax(top_scores)
    return pd.Series([top_ids[highest_index], top_scores[highest_index]])

In [112]:
def merge_score_by_date(sentiments, sent_scores):
    top = 0
    dic_count = {}
    dic_pair = {}
    for sent, sent_score in zip(sentiments, sent_scores):
        if sent not in dic_count:
            dic_count[sent] = 0
            dic_pair[sent] = []
        dic_count[sent] += 1
        dic_pair[sent].append(sent_score)

        if dic_count[sent] > top:
            top = dic_count[sent]

    top_ids = [k for k, v in dic_count.items() if v==top]
    top_scores = [sum(dic_pair[id]) for id in top_ids]
    
    if len(top_ids)==1:
        return top_ids[0]
    else:
        highest_index = np.argmax(top_scores)
        return top_ids[highest_index]

# PREPROCESSING

In [113]:
selected_sentiment_news_df = sentiment_news_df[["posted_date", "labels", "scores"]]
selected_sentiment_news_df[["sentiment", "sent_score"]] = selected_sentiment_news_df.apply(lambda x: set_sentiment(x["labels"], x["scores"]), axis=1)

C:\Users\ASUS\AppData\Local\Temp\ipykernel_2132\35320435.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sentiment_news_df[["sentiment", "sent_score"]] = selected_sentiment_news_df.apply(lambda x: set_sentiment(x["labels"], x["scores"]), axis=1)
C:\Users\ASUS\AppData\Local\Temp\ipykernel_2132\35320435.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sentiment_news_df[["sentiment", "sent_score"]] = selected_sentiment_news_df.apply(lambda x: set_sentiment(x["labels"], x["scores"]), ax

In [114]:
selected_sentiment_news_df["date"] = selected_sentiment_news_df["posted_date"].dt.date
date_sentiment_series = selected_sentiment_news_df.groupby(["date"]).apply(lambda x: merge_score_by_date(x["sentiment"], x["sent_score"]))

C:\Users\ASUS\AppData\Local\Temp\ipykernel_2132\2363464473.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_sentiment_news_df["date"] = selected_sentiment_news_df["posted_date"].dt.date
C:\Users\ASUS\AppData\Local\Temp\ipykernel_2132\2363464473.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  date_sentiment_series = selected_sentiment_news_df.groupby(["date"]).apply(lambda x: merge_score_by_date(x["sentiment"], x["sent_score"]))


In [115]:
date_sentiment_df = pd.DataFrame(date_sentiment_series)
date_sentiment_df = date_sentiment_df.reset_index()
date_sentiment_df.columns = ["time", "date_sentiment"]
# date_sentiment_df["time"] = date_sentiment_df.index
date_sentiment_df["time"] = pd.to_datetime(date_sentiment_df["time"])

In [116]:
df_merged = pd.merge(history_df, date_sentiment_df, on="time", how="left")

In [126]:
merge_save_dir = r"F:\UNIVERSITY\Project\Sentiment-Analysis-Airflow\Financial-Sentiment-Analysis\projects_financial_connect_with_sentiment\save"
df_merged.to_csv(os.path.join(merge_save_dir, "merge_df.csv"))

# EDA

In [121]:
a = [2,2,1,1,0]
counter = Counter(a)
max_count = max(counter.values())
top_ids = [id for id, count in counter.items() if count == max_count]
top_ids

[2, 1]

In [122]:
history_df

,Unnamed: 0,time,open,high,low,close,volume
0,50,2021-12-01,1478.44,1487.68,1471.30,1485.19,813267800
1,51,2021-12-02,1485.19,1493.84,1482.05,1482.05,721589500
2,52,2021-12-03,1482.05,1491.20,1443.32,1443.32,998875400
3,53,2021-12-06,1443.32,1452.55,1400.87,1413.58,978214400
4,54,2021-12-07,1413.58,1446.77,1413.58,1446.77,664287400
...,...,...,...,...,...,...,...
1029,1079,2026-01-16,1875.14,1901.22,1874.60,1879.13,1021323506
1030,1080,2026-01-19,1894.67,1898.24,1874.69,1896.59,1163188480
1031,1081,2026-01-20,1904.58,1915.11,1884.60,1893.78,1078094824
1032,1082,2026-01-21,1876.96,1894.22,1859.25,1885.44,1204631813
